In [1]:
# Import all necessary libraries
import pandas as pd              # For creating and managing data tables (DataFrames)
import numpy as np               # For numerical operations, especially with arrays
import os                        # For interacting with your computer's file system
from tqdm import tqdm            # To display cool progress bars during long operations
from sklearn.preprocessing import LabelEncoder # To convert text labels (like 'happy') into numbers
from tensorflow.keras.preprocessing.image import load_img, img_to_array # To load and convert images
from tensorflow.keras.models import Sequential # To build your neural network layer by layer
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout # Different types of layers for your CNN
from tensorflow.keras.models import load_model # To load a model you've already saved

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Define directories for training and testing data
# These paths are relative to where your Jupyter Notebook file is running from.
train_dir = 'images/train'
test_dir = 'images/test'

print(f"Expecting training data at: {os.path.abspath(train_dir)}")
print(f"Expecting testing data at: {os.path.abspath(test_dir)}")

# --- CRUCIAL: Check if the image directories actually exist ---
if not os.path.exists(train_dir):
    print(f"\nERROR: Training directory NOT FOUND at '{os.path.abspath(train_dir)}'")
    print("Please ensure your 'images' folder is placed correctly (next to your .ipynb file).")
    print("Execution stopped. Fix folder structure and restart kernel.")
    # This line stops the execution if the folder isn't found
    raise FileNotFoundError(f"Training directory not found: {os.path.abspath(train_dir)}")

if not os.path.exists(test_dir):
    print(f"\nERROR: Testing directory NOT FOUND at '{os.path.abspath(test_dir)}'")
    print("Please ensure your 'images' folder is placed correctly (next to your .ipynb file).")
    print("Execution stopped. Fix folder structure and restart kernel.")
    raise FileNotFoundError(f"Testing directory not found: {os.path.abspath(test_dir)}")

print("\nImage directories found. Proceeding with data loading.")

Expecting training data at: E:\SATHVIK\study\Projects\Face Emotion Recognition\images\train
Expecting testing data at: E:\SATHVIK\study\Projects\Face Emotion Recognition\images\test

Image directories found. Proceeding with data loading.


In [3]:
# Function to create a DataFrame containing image paths and their labels
def create_dataframe(data_dir):
    image_paths = [] # To store full paths to images
    labels = []      # To store the emotion label for each image

    # Loop through each subfolder in 'data_dir' (e.g., 'angry', 'happy')
    for label in os.listdir(data_dir):
        label_path = os.path.join(data_dir, label) # Get the full path to the emotion's folder

        # Make sure it's actually a directory before trying to list its contents
        if os.path.isdir(label_path):
            # Loop through each image file inside the emotion folder
            for image_file in os.listdir(label_path):
                image_path = os.path.join(label_path, image_file) # Get the full path to the image file
                image_paths.append(image_path) # Add the image path to our list
                labels.append(label)         # Add the corresponding emotion label

    # Create a Pandas DataFrame with two columns: 'image' (for paths) and 'label' (for emotions)
    return pd.DataFrame({'image': image_paths, 'label': labels})

print("\nCreating training DataFrame...")
train_df = create_dataframe(train_dir)
print(f"Found {len(train_df)} training images.")

print("Creating testing DataFrame...")
test_df = create_dataframe(test_dir)
print(f"Found {len(test_df)} testing images.")

# Display the first few rows of the training DataFrame to show its structure
print("\nFirst 5 rows of training DataFrame:")
print(train_df.head())


Creating training DataFrame...
Found 28821 training images.
Creating testing DataFrame...
Found 7066 testing images.

First 5 rows of training DataFrame:
                          image  label
0      images/train\angry\0.jpg  angry
1      images/train\angry\1.jpg  angry
2     images/train\angry\10.jpg  angry
3  images/train\angry\10002.jpg  angry
4  images/train\angry\10016.jpg  angry


In [4]:
# Function to extract features (load, resize, normalize images)
def extract_features(dataframe, img_size=(48, 48)):
    features = [] # List to hold the processed image data (as NumPy arrays)

    # Use tqdm to show a progress bar as we iterate through each image path
    for image_path in tqdm(dataframe['image'], desc="Extracting image features"):
        try:
            # Load the image in grayscale and resize it to 48x48 pixels
            img = load_img(image_path, color_mode='grayscale', target_size=img_size)
            img_array = img_to_array(img) # Convert the loaded image into a NumPy array
            features.append(img_array)    # Add the image array to our list
        except Exception as e:
            # If an image file is corrupted or causes an error, print a message and skip it
            print(f"Error loading image {image_path}: {e}. Skipping this image.")
            continue

    # Convert the list of individual image arrays into one large NumPy array
    features = np.array(features)
    # Normalize pixel values: Divide by 255.0 to scale them from [0, 255] to [0.0, 1.0]
    features = features.astype('float32') / 255.0
    return features

print("\nExtracting features for training data...")
# This will show a progress bar
X_train = extract_features(train_df)
print(f"Shape of X_train (Number of images, Height, Width, Channels): {X_train.shape}")

print("\nExtracting features for testing data...")
# This will also show a progress bar
X_test = extract_features(test_df)
print(f"Shape of X_test (Number of images, Height, Width, Channels): {X_test.shape}")


Extracting features for training data...


Extracting image features: 100%|████████████████████████████████████████████████████████████████████████████| 28821/28821 [00:07<00:00, 4066.41it/s]


Shape of X_train (Number of images, Height, Width, Channels): (28821, 48, 48, 1)

Extracting features for testing data...


Extracting image features: 100%|██████████████████████████████████████████████████████████████████████████████| 7066/7066 [00:01<00:00, 4017.09it/s]

Shape of X_test (Number of images, Height, Width, Channels): (7066, 48, 48, 1)


In [5]:
# Encode labels (convert string labels to numerical format)
le = LabelEncoder() # Create an instance of the LabelEncoder

# Fit and transform training labels: This step learns the unique labels and assigns a number to each.
y_train = le.fit_transform(train_df['label'])
# Transform testing labels: Use the SAME mapping learned from the training data.
y_test = le.transform(test_df['label'])

# Reshape labels: Add an extra dimension (from (N,) to (N, 1)) to match the expected input shape for Keras's
# sparse_categorical_crossentropy loss function.
y_train = np.expand_dims(y_train, -1)
y_test = np.expand_dims(y_test, -1)

# Get the total number of unique emotion classes
num_classes = len(le.classes_)
print(f"\nDetected {num_classes} emotion classes:")
# Print the mapping from the assigned number to the actual emotion label
for i, emotion in enumerate(le.classes_):
    print(f"  {i}: {emotion}")

print(f"Shape of y_train (Number of labels, 1): {y_train.shape}")
print(f"Shape of y_test (Number of labels, 1): {y_test.shape}")


Detected 7 emotion classes:
  0: angry
  1: disgust
  2: fear
  3: happy
  4: neutral
  5: sad
  6: surprise
Shape of y_train (Number of labels, 1): (28821, 1)
Shape of y_test (Number of labels, 1): (7066, 1)


In [6]:
# Build the Convolutional Neural Network (CNN) model
model = Sequential([
    # Input Layer: Explicitly defines the shape of our input images (48x48 pixels, 1 channel for grayscale).
    # This removes a common Keras UserWarning.
    Input(shape=(48, 48, 1)),

    # First Convolutional Block
    # Conv2D: This layer learns different features (like edges, textures) from the images.
    #   32: Number of filters/feature detectors.
    #   (3, 3): Size of each filter (3x3 pixels).
    #   'relu': Rectified Linear Unit activation function (introduces non-linearity).
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)), # MaxPooling2D: Reduces the spatial dimensions (height/width) of the feature maps,
                          # keeping the most important information, and makes the model more robust to shifts.
    Dropout(0.25),        # Dropout: Randomly sets a fraction of neurons to zero during training.
                          # This helps prevent the model from memorizing the training data too well (overfitting).

    # Second Convolutional Block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Third Convolutional Block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Flatten(), # Flatten: Converts the 2D feature maps from the convolutional layers into a 1D vector.
               # This prepares the data for the fully connected (Dense) layers.

    # Fully Connected (Dense) Layers
    Dense(1024, activation='relu'), # Dense layer: A standard neural network layer where every input neuron is connected
                                    # to every output neuron. 1024 neurons here.
    Dropout(0.5),                   # Another Dropout layer to further prevent overfitting.

    # Output Layer
    # Dense: The final layer.
    #   num_classes: Number of neurons equals the number of emotion categories (e.g., 7 for angry, happy, etc.).
    #   'softmax' activation: Outputs a probability distribution over the 'num_classes'. The sum of probabilities for
    #                         all emotions will be 1, indicating the model's confidence for each emotion.
    Dense(num_classes, activation='softmax')
])

# Print a summary of the model architecture, showing the layers, their output shapes, and the number of parameters.
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 46, 46, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 23, 23, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 23, 23, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 21, 21, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 10, 10, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 10, 10, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 8, 8, 128)           │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 4, 4, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 4, 4, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2048)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1024)                │       2,098,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 7)                   │           7,175 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,198,023 (8.38 MB)

 Trainable params: 2,198,023 (8.38 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Compile the model
# optimizer='adam': An efficient and popular optimization algorithm for training neural networks.
# loss='sparse_categorical_crossentropy': This loss function is suitable for classification problems
#                                        where your labels are integers (which they are after LabelEncoding).
# metrics=['accuracy']: During training, we want to see how accurate our model is.
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\nModel compiled successfully!")


Model compiled successfully!


In [8]:
# --- Step 8: Train the Model (UNCOMMENT TO RUN) ---
# This is the step that performs the actual learning. It can take a long time to complete!
# The 'history' object will store training metrics like loss and accuracy for each epoch.

print("\n--- Starting Model Training ---")
print("This will take a while. Please do not interrupt it.")

# --- UNCOMMENT THE NEXT LINE TO START TRAINING! ---
history = model.fit(X_train, y_train,        # Your training data (features and labels)
                   epochs=50,                # Number of times the model will see the entire training dataset
                      batch_size=64,            # Number of samples processed before the model's weights are updated
                     validation_data=(X_test, y_test)) # Data used to evaluate the model's performance after each epoch (validation set)

print("Model training complete (if uncommented and run).")

# You can optionally run this after training to see the keys available in the history object
if 'history' in locals():
     print(history.history.keys())


--- Starting Model Training ---
This will take a while. Please do not interrupt it.
Epoch 1/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 29s 61ms/step - accuracy: 0.2486 - loss: 1.8079 - val_accuracy: 0.3899 - val_loss: 1.5707
Epoch 2/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 29s 64ms/step - accuracy: 0.3986 - loss: 1.5498 - val_accuracy: 0.4551 - val_loss: 1.4364
Epoch 3/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 32s 71ms/step - accuracy: 0.4492 - loss: 1.4231 - val_accuracy: 0.4966 - val_loss: 1.3164
Epoch 4/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 28s 63ms/step - accuracy: 0.4816 - loss: 1.3452 - val_accuracy: 0.5273 - val_loss: 1.2584
Epoch 5/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 31s 70ms/step - accuracy: 0.5071 - loss: 1.2876 - val_accuracy: 0.5347 - val_loss: 1.2275
Epoch 6/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 33s 73ms/step - accuracy: 0.5287 - loss: 1.2403 - val_accuracy: 0.5511 - val_loss: 1.1859
Epoch 7/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 33s 73ms/step - accuracy: 0.5475 - loss: 1.1904 - val_accuracy: 0.5654 - val_loss: 1.1632
Epoch

In [11]:
# --- Step 9: Save the Trained Model (UNCOMMENT TO RUN) ---
# This saves the model's learned weights and architecture to a file named 'emotion_recognition_model.h5'.
# This file will be saved in the SAME directory as your Jupyter Notebook.

print("\nAttempting to save the trained model...")

# --- UNCOMMENT THE NEXT LINE TO SAVE THE MODEL! ---
model.save('emotion_recognition_model.h5')

print("Model save command executed (check for file).")



Attempting to save the trained model...
Model save command executed (check for file).


In [18]:
# --- Step 10: Load and Use the Saved Model (Example) ---
print("\n--- Model Prediction Example ---")

try:
    #Attempt to load the model from the .h5 file
    loaded_model = load_model('emotion_recognition_model.h5')
    print("Model 'emotion_recognition_model.h5' loaded successfully for prediction.")
except Exception as e:
    # If the file isn't found or there's an issue loading, print an error
    print(f"Error loading model: {e}. Please ensure 'emotion_recognition_model.h5' exists (check its location and if it was saved).")
    loaded_model = None # Set to None so we don't try to use a non-existent model

if loaded_model: # Only proceed if the model was loaded successfully
    # Ensure emotion_labels matches the order from your LabelEncoder (le.classes_).
    # If 'le' object is still defined from Step 5, this is the most reliable way:
    emotion_labels = list(le.classes_)
    # Otherwise, manually define them in the order you know they were encoded:
    emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

    # Function to predict emotion from a single image file
    def predict_emotion(image_path, model, labels, img_size=(48, 48)):
        try:
            # Load, grayscale, and resize the image
            img = load_img(image_path, color_mode='grayscale', target_size=img_size)
            img_array = img_to_array(img)
            # Add a "batch" dimension to the image array (model expects a batch, even if it's just one image)
            processed_img = np.expand_dims(img_array, axis=0) # Shape becomes (1, 48, 48, 1)
            # Normalize pixel values
            processed_img = processed_img.astype('float32') / 255.0

            # Get predictions (probabilities for each emotion)
            predictions = model.predict(processed_img, verbose=0)[0] # verbose=0 suppresses progress bar for single prediction
            # Get the index of the emotion with the highest probability
            predicted_label_index = np.argmax(predictions)
            # Map the index back to the emotion label (e.g., 0 -> 'angry')
            predicted_label = labels[predicted_label_index]
            return predicted_label
        except Exception as e:
            print(f"Error predicting emotion for {image_path}: {e}")
            return "Prediction Error"

    # IMPORTANT: Replace 'images/train/happy/0.jpg' with an actual path to an image file that EXISTS in your dataset!
    # For example, if you have an image at 'images/test/angry/test_angry_123.jpg', use that path.
    sample_image_path = 'images/train/happy/7.jpg' # <--- CHANGE THIS PATH TO A REAL IMAGE YOU HAVE!

    if os.path.exists(sample_image_path):
        predicted_emotion = predict_emotion(sample_image_path, loaded_model, emotion_labels)
        print(f"Predicted emotion for {sample_image_path}: {predicted_emotion}")
    else:
        print(f"Sample image not found at: {sample_image_path}. Please provide a valid path to an image you have.")
else:
    print("Skipping prediction example because the model could not be loaded successfully.")


--- Model Prediction Example ---
Model 'emotion_recognition_model.h5' loaded successfully for prediction.
Predicted emotion for images/train/happy/7.jpg: happy
